# Chapter Title Generator

**Michael Zeng, Andrew Wang, Lakshana Kathirkamaranjan, Peri Gandhi**

*Supervised BERT Encoder–Decoder (seq2seq) system that suggests creative chapter titles from narrative text.*

---

> **Submission note:** This notebook contains the full project report (text and figures) plus a complete copy of all Python source files. The standalone `.py` modules in the repository remain the primary runnable entry points. Code cells below the **Code Appendix** divider are copies for grading and PDF export; page limits apply to narrative content only.


## 1. Problem Statement

Authors often struggle to name chapters in a way that fits tone, genre, and narrative events. Naming can interrupt creative flow and is especially difficult for new writers who lack established conventions or confidence.

We address this with a **supervised sequence-to-sequence model** that reads a passage of narrative text and proposes one or more **title candidates**. The system is instruction-tuned: the encoder receives a prefixed prompt (`generate chapter title: ...`) and the decoder generates a short title token-by-token.

**Goals:**
- Reduce friction during the writing process by offering multiple creative options.
- Learn mappings from narrative content to concise, human-like titles.
- Support multiple training corpora so team members can experiment with different text domains.


## 2. Approach

We fine-tune a **BERT Encoder–Decoder** (`EncoderDecoderModel` with `google-bert/bert-base-uncased`) using Hugging Face `Seq2SeqTrainer`.

| Component | Role |
|-----------|------|
| **Encoder** | Reads instruction-tuned narrative text and produces a contextual representation |
| **Decoder** | Generates the title autoregressively |
| **Beam search (inference)** | Returns multiple diverse title candidates |

**Instruction format**

```
Input:  generate chapter title: <narrative text>
Output: <title>
```

**Training highlights** (see `config.py`):
- Max input length 384 tokens; max target length 48 tokens
- Batch size 1 with gradient accumulation (effective batch 8)
- FP16 + gradient checkpointing for GPU memory efficiency
- Profanity filtering on training pairs and blocked-token masking at generation time


## 3. System Architecture

The pipeline has three stages: **data preparation**, **fine-tuning**, and **inference**.

In [ ]:
# Figure 1 — End-to-end pipeline
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 4))
ax.set_xlim(0, 11)
ax.set_ylim(0, 4)
ax.axis('off')

boxes = [
    (0.3, 1.6, 2.2, 1.2, 'Datasets\n(CMU / Novel Chapter /\nSimpleStories)', '#dbeafe'),
    (3.0, 1.6, 2.0, 1.2, 'JSONL\npreprocessing\n+ safety filter', '#dcfce7'),
    (5.5, 1.6, 2.3, 1.2, 'BERT Encoder–Decoder\nfine-tuning\n(Seq2SeqTrainer)', '#fde68a'),
    (8.2, 1.6, 2.3, 1.2, 'Beam search\ntitle candidates\n(at inference)', '#fce7f3'),
]
for x, y, w, h, text, color in boxes:
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.05',
        linewidth=1.5,
        edgecolor='#334155',
        facecolor=color,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=10)

for x1, x2 in [(2.5, 3.0), (5.0, 5.5), (7.8, 8.2)]:
    ax.annotate('', xy=(x2, 2.2), xytext=(x1, 2.2), arrowprops=dict(arrowstyle='->', lw=1.8))

ax.text(5.5, 3.5, 'Figure 1. Chapter Title Generator pipeline', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Datasets

Each team member contributed a dataset loader. All loaders write JSONL files under `data/` with fields `text` (prefixed narrative) and `title` (target label).

| Team member | Script | Source | Mapping | Approx. size |
|-------------|--------|--------|---------|--------------|
| **Michael Zeng** | `data_cmu_book_summaries.py` | [CMU Book Summaries](https://huggingface.co/datasets/textminr/cmu-book-summaries) | plot summary → book title | ~16.6k |
| **Andrew Wang** | `data_novel_chapter.py` | [Novel Chapter / BookSum](https://github.com/manestay/novel-chapter-dataset) | study-guide summary → section title | ~9.6k train (BookSum) |
| **Michael Zeng** | `data_simple_stories.py` | [SimpleStories](https://huggingface.co/datasets/SimpleStories/SimpleStories) | short story → theme label | 20k cap (2.1M total) |

**CMU Book Summaries** provides book-level plot summaries paired with published titles—useful for learning high-level thematic naming.

**Novel Chapter Dataset** (via BookSum on Hugging Face) pairs classic literature chapter summaries with section names, supporting extraction of key themes from longer-form narrative summaries.

**SimpleStories** contains over two million model-generated short stories with rich metadata (`theme`, `topic`, `style`). We map **story → theme** to train on simple narrative structure and key-idea retrieval from short texts. Streaming load caps at 20,000 rows by default for practical training.

`train.py` uses **CMU Book Summaries** by default; other JSONL files can be swapped in for domain-specific experiments.


In [ ]:
# Figure 2 — Dataset scale comparison (rows used in this project)
import matplotlib.pyplot as plt

names = ['CMU Book\nSummaries', 'Novel Chapter\n(BookSum train)', 'SimpleStories\n(20k cap)']
sizes = [16600, 9600, 20000]
colors = ['#3b82f6', '#10b981', '#f59e0b']

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(names, sizes, color=colors, edgecolor='#1e293b')
ax.set_ylabel('Training pairs (approx.)')
ax.set_title('Figure 2. Dataset sizes used in the project')
ax.bar_label(bars, fmt='{:,.0f}', padding=3)
plt.tight_layout()
plt.show()

## 5. Data Preprocessing

Shared utilities in `data_utils.py`:
- **Chunking** — truncate narratives to `CHUNK_CHARS` (3,000) before tokenization
- **Instruction prefix** — prepend `generate chapter title: `
- **Deduplication** — remove duplicate (text, title) pairs
- **Train/val split** — 90/10 with configurable caps (`MAX_TRAIN_SAMPLES`, `MAX_VAL_SAMPLES`)

Dataset-specific filters remove low-quality titles (e.g., generic "Chapter 3" for CMU), enforce minimum text length, and apply the safety module.


## 6. Safety

Creative writing tools should avoid offensive outputs. We implement a two-layer safety strategy:

1. **Training-time filtering** (`safety.py`) — `better-profanity` plus a regex blocklist scan both narrative text and target titles; unsafe pairs are dropped.
2. **Generation-time masking** (`generate_title.py`) — a custom `LogitsProcessor` sets logits of blocked vocabulary tokens to negative infinity during beam search.

This reduces exposure to profanity in both training data and model outputs.


## 7. Model & Training

| Hyperparameter | Value |
|----------------|-------|
| Encoder / Decoder | `google-bert/bert-base-uncased` |
| Epochs | 1 (smoke-test default; increase for full runs) |
| Batch size | 1 (×8 gradient accumulation) |
| Learning rate | 5e-5 |
| Max input / target length | 384 / 48 |
| FP16 | Enabled when CUDA available |

Training uses `Seq2SeqTrainer` with epoch-level evaluation, best-checkpoint selection by `eval_loss`, and checkpoint export to `checkpoints/bert2bert-titles/best/`.

**Hardware note:** `bert-base` needs ~6 GB+ VRAM. On smaller GPUs, switch to `prajjwal1/bert-tiny` in `config.py`.


## 8. Inference & Evaluation

`generate_title.py` supports three modes:
- `--text "..."` — generate titles from inline narrative text
- `--chapter-file path.txt` — read narrative from a file
- `--eval` — exact-match accuracy on the CMU validation JSONL

Beam search returns **multiple candidates** (`NUM_CANDIDATES=3` by default) so authors can choose among options.

**Evaluation metric:** exact string match (case-insensitive) between the top prediction and gold title on the validation set. This is a strict metric; partial matches and semantic similarity are left for future work.


## 9. Results & Discussion

We built and validated the full pipeline: dataset loaders for three corpora, JSONL export, BERT seq2seq training script, and inference with safety masking.

Title generation is inherently subjective—many valid titles exist for the same passage—so exact-match accuracy underestimates usefulness. The multi-candidate beam search interface is designed for **human-in-the-loop** selection rather than single correct answers.

**Observations:**
- CMU summaries → book titles teach high-level thematic naming.
- Novel Chapter summaries → section titles expose the model to classic literature structure.
- SimpleStories story → theme pairs emphasize concise key-idea extraction from short fiction.

**Limitations:** small default training caps for fast iteration; BERT-base memory requirements; exact-match eval ignores semantic equivalence; model quality depends on GPU training completing successfully.

**Future work:** combine datasets during training, add BLEU/ROUGE or embedding-based metrics, fine-tune on user-specific manuscripts, and deploy a simple web UI.


## 10. Conclusion

We present a BERT encoder–decoder chapter title generator with three dataset pipelines, profanity-safe training and inference, and multi-candidate beam search output. The modular design lets each team member experiment with a different narrative domain while sharing the same training and generation stack.

---

## References

1. CMU Book Summaries — https://huggingface.co/datasets/textminr/cmu-book-summaries
2. Ladhak et al. (2020). *Exploring Content Selection in Summarization of Novel Chapters.* ACL.
3. Kryściński et al. (2021). *BookSum: A Collection of Datasets for Long-form Narrative Summarization.*
4. Finke et al. (2025). *Parameterized Synthetic Text Generation with SimpleStories.* arXiv:2504.09184
5. Devlin et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.*
6. Hugging Face `EncoderDecoderModel` and `Seq2SeqTrainer` documentation

---

# Code Appendix

Complete copies of project source files follow. Run from the repository root using the standalone `.py` files (e.g. `python train.py`).


### `requirements.txt`

In [ ]:
torch>=2.0.0
# For GPU training on Windows/NVIDIA, install CUDA PyTorch separately:
#   pip install torch --index-url https://download.pytorch.org/whl/cu124
transformers>=4.40.0
datasets>=2.14.0
accelerate>=0.27.0
sentencepiece>=0.1.99
better-profanity>=0.7.0
dill>=0.3.8


### `config.py`

In [ ]:
"""
BERT Encoder–Decoder chapter title generator (seq2seq).

Fine-tunes Hugging Face EncoderDecoderModel (BERT encoder + BertGeneration decoder).
"""

# Model — bert-base + fp16 fits most 4 GB GPUs; lower MAX_INPUT_LENGTH if OOM
ENCODER_MODEL = "google-bert/bert-base-uncased"
DECODER_MODEL = "google-bert/bert-base-uncased"
CHECKPOINT_DIR = "checkpoints/bert2bert-titles"

# Instruction-tuning prefix (encoder input)
INPUT_PREFIX = "generate chapter title: "

# Sequence lengths
MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 48

# Training
# Use a short first pass for smoke testing; raise these later for full training.
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
FP16 = True
GRADIENT_CHECKPOINTING = True

# Dataset — CMU Book Summaries (summary text → book title)
DATASET_ID = "textminr/cmu-book-summaries"
VAL_FRACTION = 0.1
RANDOM_SEED = 42
MAX_ROWS = None  # None = use full dataset (~16.6k rows)
MAX_TRAIN_SAMPLES = 2000
MAX_VAL_SAMPLES = 200
CHUNK_CHARS = 3_000

# Generation
NUM_BEAMS = 4
NUM_CANDIDATES = 3
MAX_GEN_LENGTH = 48

# Safety
ENABLE_PROFANITY_FILTER = True
BLOCKED_WORDS = (
    "damn",
    "hell",
    "shit",
    "fuck",
    "bitch",
    "asshole",
    "bastard",
)


### `data_utils.py`

In [ ]:
"""Shared helpers for dataset loaders and inference."""

from __future__ import annotations

import json
from pathlib import Path

import config as cfg


def chunk_text(text: str, max_chars: int | None = None) -> str:
    max_chars = max_chars if max_chars is not None else cfg.CHUNK_CHARS
    text = text.strip()
    return text if len(text) <= max_chars else text[:max_chars]


def format_input(text: str) -> str:
    return cfg.INPUT_PREFIX + chunk_text(text)


def _row_key(row: dict) -> tuple[str, str]:
    return (row["text"].strip().lower(), row["title"].strip().lower())


def dedupe_rows(rows: list[dict]) -> list[dict]:
    seen: set[tuple[str, str]] = set()
    unique: list[dict] = []
    for row in rows:
        key = _row_key(row)
        if key in seen:
            continue
        seen.add(key)
        unique.append(row)
    return unique


def export_jsonl(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


### `safety.py`

In [ ]:
"""Profanity / blocked-word filtering for training data and generation."""

from __future__ import annotations

import re

from config import BLOCKED_WORDS, ENABLE_PROFANITY_FILTER

try:
    from better_profanity import profanity

    profanity.load_censor_words()
    _HAS_PROFANITY = True
except ImportError:
    _HAS_PROFANITY = False

# better_profanity is very slow on long text; use it only for short strings
_MAX_PROFANITY_CHARS = 512
_BLOCK_RE = re.compile(
    r"\b(" + "|".join(re.escape(w) for w in BLOCKED_WORDS) + r")\b",
    re.I,
)


def contains_blocked_text(text: str) -> bool:
    if not ENABLE_PROFANITY_FILTER or not text:
        return False
    if _BLOCK_RE.search(text):
        return True
    if _HAS_PROFANITY and len(text) <= _MAX_PROFANITY_CHARS:
        return profanity.contains_profanity(text)
    return False


def is_safe_pair(text: str, title: str) -> bool:
    return not contains_blocked_text(text) and not contains_blocked_text(title)


def blocked_word_token_ids(tokenizer) -> set[int]:
    """Token ids whose decoded form is a blocked word (for logits masking)."""
    ids: set[int] = set()
    if not ENABLE_PROFANITY_FILTER:
        return ids
    for token, idx in tokenizer.get_vocab().items():
        decoded = token.replace("##", "").lower()
        if decoded in BLOCKED_WORDS:
            ids.add(idx)
    return ids


### `data_cmu_book_summaries.py`

In [ ]:
"""
Load CMU Book Summaries for BERT seq2seq title generation.

Dataset: textminr/cmu-book-summaries (Hugging Face)
  https://huggingface.co/datasets/textminr/cmu-book-summaries

~16.6k book plot summaries paired with book titles (summary → title).
"""

from __future__ import annotations

import random
import re
from pathlib import Path

import config as cfg
from data_utils import chunk_text, dedupe_rows, export_jsonl, format_input
from safety import is_safe_pair

DATASET_ID = "textminr/cmu-book-summaries"
TRAIN_JSONL = "seq2seq_train_cmu_book_summaries.jsonl"
VAL_JSONL = "seq2seq_val_cmu_book_summaries.jsonl"
PLAY_WORDS = ("scene", "act", "prologue", "epilogue")
GENERIC_CHAPTER = re.compile(r"^chapters?\s+(\d+|[ivxlcdm]+)$", re.I)
MIN_SUMMARY_CHARS = 100


def _title_ok(title: str) -> bool:
    title = title.strip()
    if len(title) < 2 or len(title) > 120:
        return False
    lower = title.lower()
    if any(w in lower for w in PLAY_WORDS):
        return False
    if GENERIC_CHAPTER.match(lower):
        return False
    if lower.startswith("chapter "):
        return False
    return True


def load_cmu_book_summaries(max_rows: int | None = None) -> list[dict]:
    from datasets import load_dataset

    print(f"  loading {DATASET_ID}...", flush=True)
    ds = load_dataset(DATASET_ID, split="train")
    rows: list[dict] = []

    for i, row in enumerate(ds, 1):
        title = (row.get("title") or "").strip()
        summary = (row.get("summary") or "").strip()
        if len(summary) < MIN_SUMMARY_CHARS or not _title_ok(title):
            continue
        chunk = chunk_text(summary)
        if not is_safe_pair(chunk, title):
            continue
        rows.append(
            {
                "text": format_input(summary),
                "title": title,
                "source": "cmu_book_summaries",
            }
        )
        if max_rows and len(rows) >= max_rows:
            break
        if i % 2000 == 0:
            print(f"    processed {i:,}, kept {len(rows):,}", flush=True)

    return rows


def build_dataset(root: Path | None = None) -> tuple[list[dict], list[dict]]:
    root = root or Path(__file__).parent

    print("Loading CMU Book Summaries...")
    all_rows = load_cmu_book_summaries(cfg.MAX_ROWS)
    print(f"  {len(all_rows):,} rows after filtering")

    before = len(all_rows)
    all_rows = dedupe_rows(all_rows)
    if before != len(all_rows):
        print(f"  deduplicated {before:,} -> {len(all_rows):,} rows")

    if not all_rows:
        raise SystemExit(f"No rows loaded from {DATASET_ID}")

    random.seed(cfg.RANDOM_SEED)
    random.shuffle(all_rows)

    val_size = max(1, int(len(all_rows) * cfg.VAL_FRACTION))
    if cfg.MAX_VAL_SAMPLES:
        val_size = min(val_size, cfg.MAX_VAL_SAMPLES)
    val_rows = all_rows[:val_size]
    train_rows = all_rows[val_size:]

    if cfg.MAX_TRAIN_SAMPLES:
        train_rows = train_rows[: cfg.MAX_TRAIN_SAMPLES]

    return train_rows, val_rows


def prepare_data(root: Path | None = None) -> tuple[Path, Path]:
    root = root or Path(__file__).parent
    train_rows, val_rows = build_dataset(root)
    train_path = root / "data" / TRAIN_JSONL
    val_path = root / "data" / VAL_JSONL
    export_jsonl(train_rows, train_path)
    export_jsonl(val_rows, val_path)
    print(f"Wrote {len(train_rows):,} train -> {train_path}")
    print(f"Wrote {len(val_rows):,} val   -> {val_path}")
    return train_path, val_path


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Prepare CMU Book Summaries (textminr/cmu-book-summaries on Hugging Face)"
    )
    parser.add_argument(
        "--quick",
        action="store_true",
        help="Use only 500 rows for a fast smoke test",
    )
    args = parser.parse_args()
    if args.quick:
        cfg.MAX_ROWS = 500
        cfg.MAX_TRAIN_SAMPLES = 450
        cfg.MAX_VAL_SAMPLES = 50
    prepare_data()


### `data_novel_chapter.py`

In [ ]:
"""
Load Novel Chapter Dataset for BERT seq2seq title generation.

Dataset: manestay/novel-chapter-dataset (GitHub)
  https://github.com/manestay/novel-chapter-dataset

Study-guide chapter summaries paired with Project Gutenberg chapter text
(Ladhak et al., ACL 2020). Summary pickle files are not hosted on GitHub;
by default this loader uses kmfoda/booksum on Hugging Face (BookSum,
Kryściński et al., 2021), which packages the same chapter-summary pairs.

For fully local loading, run make_data_splits.py in the upstream repo and pass
--pickle-dir pointing at the raw_splits/ folder.
"""

from __future__ import annotations

import json
import random
import urllib.request
from pathlib import Path

import config as cfg
from data_utils import chunk_text, dedupe_rows, export_jsonl, format_input
from safety import is_safe_pair

BOOKSUM_DATASET_ID = "kmfoda/booksum"
TRAIN_JSONL = "seq2seq_train_novel_chapter.jsonl"
VAL_JSONL = "seq2seq_val_novel_chapter.jsonl"
NOVEL_CHAPTER_REPO = "https://github.com/manestay/novel-chapter-dataset"
RAW_TEXTS_URL = (
    "https://github.com/manestay/novel-chapter-dataset/raw/main/pks/raw_texts.pk"
)
MIN_TEXT_CHARS = 100


def _section_title_ok(title: str) -> bool:
    title = title.strip()
    return 2 <= len(title) <= 120


def _summary_to_text(summary: object) -> str:
    if isinstance(summary, list):
        parts: list[str] = []
        for item in summary:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, list):
                parts.extend(str(x) for x in item)
        return " ".join(parts).strip()
    return str(summary or "").strip()


def _raw_text_to_str(raw_text: object) -> str:
    if isinstance(raw_text, list):
        return " ".join(str(x) for x in raw_text).strip()
    return str(raw_text or "").strip()


def _section_title_from_id(sect_id: str) -> str:
    return sect_id.rsplit(".", 1)[-1].strip()


def _append_row(
    rows: list[dict],
    narrative: str,
    title: str,
    source: str,
    max_rows: int | None,
) -> bool:
    narrative = narrative.strip()
    title = title.strip()
    if len(narrative) < MIN_TEXT_CHARS or not _section_title_ok(title):
        return False
    chunk = chunk_text(narrative)
    if not is_safe_pair(chunk, title):
        return False
    rows.append(
        {
            "text": format_input(narrative),
            "title": title,
            "source": source,
        }
    )
    if max_rows and len(rows) >= max_rows:
        return True
    return False


def load_from_booksum(
    max_rows: int | None = None,
    *,
    use_summary: bool = True,
) -> list[dict]:
    from datasets import load_dataset

    print(f"  loading {BOOKSUM_DATASET_ID} (Novel Chapter / BookSum)...", flush=True)
    ds = load_dataset(BOOKSUM_DATASET_ID, split="train")
    rows: list[dict] = []

    for i, row in enumerate(ds, 1):
        title = (row.get("summary_name") or "").strip()
        if use_summary:
            narrative = (row.get("summary_text") or "").strip()
        else:
            narrative = (row.get("chapter") or "").strip()
        if _append_row(rows, narrative, title, "novel_chapter_booksum", max_rows):
            break
        if i % 2000 == 0:
            print(f"    processed {i:,}, kept {len(rows):,}", flush=True)

    return rows


def _load_pickle(path: Path):
    import pickle

    try:
        import dill  # noqa: F401
    except ImportError as exc:
        raise SystemExit(
            "Pickle loading requires dill. Install with: pip install dill"
        ) from exc

    import dill

    with path.open("rb") as f:
        return dill.load(f)


def load_from_pickles(
    pickle_dir: Path,
    max_rows: int | None = None,
    *,
    use_summary: bool = True,
) -> list[dict]:
    pickle_dir = pickle_dir.resolve()
    pk_files = sorted(pickle_dir.glob("*.pk"))
    if not pk_files:
        raise SystemExit(f"No .pk files found in {pickle_dir}")

    rows: list[dict] = []
    for pk_path in pk_files:
        print(f"  loading {pk_path.name}...", flush=True)
        split_rows = _load_pickle(pk_path)
        for item in split_rows:
            sect_id = (item.get("id") or "").strip()
            title = _section_title_from_id(sect_id) if sect_id else ""
            if use_summary:
                summaries = item.get("summaries") or []
                if not summaries:
                    continue
                narrative = _summary_to_text(summaries[0].get("summary"))
            else:
                narrative = _raw_text_to_str(item.get("raw_text"))
            if _append_row(rows, narrative, title, "novel_chapter_pickle", max_rows):
                return rows

    return rows


def download_raw_texts_pk(dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return dest
    print(f"  downloading raw_texts.pk from {NOVEL_CHAPTER_REPO}...", flush=True)
    urllib.request.urlretrieve(RAW_TEXTS_URL, dest)
    return dest


def load_from_raw_texts(
    raw_texts_path: Path | None = None,
    max_rows: int | None = None,
) -> list[dict]:
    root = Path(__file__).parent
    raw_texts_path = raw_texts_path or root / "data" / "novel-chapter" / "raw_texts.pk"
    download_raw_texts_pk(raw_texts_path)
    raw_texts = _load_pickle(raw_texts_path)
    rows: list[dict] = []

    for book_title, chapters in raw_texts.items():
        for chapter_name, chapter_lines in chapters.items():
            narrative = _raw_text_to_str(chapter_lines)
            title = chapter_name.strip()
            if _append_row(rows, narrative, title, "novel_chapter_gutenberg", max_rows):
                return rows

    return rows


def load_novel_chapter_dataset(
    max_rows: int | None = None,
    *,
    pickle_dir: Path | None = None,
    use_summary: bool = True,
    raw_texts_only: bool = False,
) -> list[dict]:
    if raw_texts_only:
        return load_from_raw_texts(max_rows=max_rows)
    if pickle_dir is not None:
        return load_from_pickles(pickle_dir, max_rows, use_summary=use_summary)
    return load_from_booksum(max_rows, use_summary=use_summary)


def build_dataset(
    root: Path | None = None,
    *,
    pickle_dir: Path | None = None,
    use_summary: bool = True,
    raw_texts_only: bool = False,
) -> tuple[list[dict], list[dict]]:
    root = root or Path(__file__).parent

    print("Loading Novel Chapter Dataset...")
    all_rows = load_novel_chapter_dataset(
        cfg.MAX_ROWS,
        pickle_dir=pickle_dir,
        use_summary=use_summary,
        raw_texts_only=raw_texts_only,
    )
    print(f"  {len(all_rows):,} rows after filtering")

    before = len(all_rows)
    all_rows = dedupe_rows(all_rows)
    if before != len(all_rows):
        print(f"  deduplicated {before:,} -> {len(all_rows):,} rows")

    if not all_rows:
        raise SystemExit("No rows loaded from Novel Chapter Dataset")

    random.seed(cfg.RANDOM_SEED)
    random.shuffle(all_rows)

    val_size = max(1, int(len(all_rows) * cfg.VAL_FRACTION))
    if cfg.MAX_VAL_SAMPLES:
        val_size = min(val_size, cfg.MAX_VAL_SAMPLES)
    val_rows = all_rows[:val_size]
    train_rows = all_rows[val_size:]

    if cfg.MAX_TRAIN_SAMPLES:
        train_rows = train_rows[: cfg.MAX_TRAIN_SAMPLES]

    return train_rows, val_rows


def prepare_data(
    root: Path | None = None,
    *,
    pickle_dir: Path | None = None,
    use_summary: bool = True,
    raw_texts_only: bool = False,
) -> tuple[Path, Path]:
    root = root or Path(__file__).parent
    train_rows, val_rows = build_dataset(
        root,
        pickle_dir=pickle_dir,
        use_summary=use_summary,
        raw_texts_only=raw_texts_only,
    )
    train_path = root / "data" / TRAIN_JSONL
    val_path = root / "data" / VAL_JSONL
    export_jsonl(train_rows, train_path)
    export_jsonl(val_rows, val_path)
    print(f"Wrote {len(train_rows):,} train -> {train_path}")
    print(f"Wrote {len(val_rows):,} val   -> {val_path}")
    return train_path, val_path


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Prepare Novel Chapter Dataset (manestay/novel-chapter-dataset)"
    )
    parser.add_argument(
        "--quick",
        action="store_true",
        help="Use only 500 rows for a fast smoke test",
    )
    parser.add_argument(
        "--pickle-dir",
        type=Path,
        default=None,
        help="Path to manestay raw_splits/ folder with train.pk / val.pk / test.pk",
    )
    parser.add_argument(
        "--chapter-text",
        action="store_true",
        help="Use full chapter text as input instead of study-guide summaries",
    )
    parser.add_argument(
        "--raw-texts-only",
        action="store_true",
        help="Use only Project Gutenberg chapter text from raw_texts.pk (no summaries)",
    )
    args = parser.parse_args()
    if args.quick:
        cfg.MAX_ROWS = 500
        cfg.MAX_TRAIN_SAMPLES = 450
        cfg.MAX_VAL_SAMPLES = 50
    prepare_data(
        pickle_dir=args.pickle_dir,
        use_summary=not args.chapter_text,
        raw_texts_only=args.raw_texts_only,
    )


### `data_simple_stories.py`

In [ ]:
"""
Load SimpleStories for BERT seq2seq title generation.

Dataset: SimpleStories/SimpleStories (Hugging Face)
  https://huggingface.co/datasets/SimpleStories/SimpleStories

Model-generated short stories annotated with theme, topic, style, etc.
Uses story text as input and theme label as the target title for learning
key ideas from short narrative text.
"""

from __future__ import annotations

import random
from pathlib import Path

import config as cfg
from data_utils import chunk_text, dedupe_rows, export_jsonl, format_input
from safety import is_safe_pair

DATASET_ID = "SimpleStories/SimpleStories"
TRAIN_JSONL = "seq2seq_train_simple_stories.jsonl"
VAL_JSONL = "seq2seq_val_simple_stories.jsonl"
DEFAULT_MAX_ROWS = 20_000  # cap streaming scan (full dataset is ~2.1M)
MIN_STORY_CHARS = 80
MIN_WORD_COUNT = 40


def _theme_title_ok(title: str) -> bool:
    title = title.strip()
    return 2 <= len(title) <= 80


def load_simple_stories(max_rows: int | None = None) -> list[dict]:
    from datasets import load_dataset

    print(f"  loading {DATASET_ID}...", flush=True)
    ds = load_dataset(DATASET_ID, split="train", streaming=True)
    rows: list[dict] = []

    for i, row in enumerate(ds, 1):
        story = (row.get("story") or "").strip()
        theme = (row.get("theme") or "").strip()
        word_count = row.get("word_count") or 0

        if len(story) < MIN_STORY_CHARS or word_count < MIN_WORD_COUNT:
            continue
        if not _theme_title_ok(theme):
            continue

        chunk = chunk_text(story)
        if not is_safe_pair(chunk, theme):
            continue

        rows.append(
            {
                "text": format_input(story),
                "title": theme,
                "source": "simple_stories",
                "topic": (row.get("topic") or "").strip(),
            }
        )
        if max_rows and len(rows) >= max_rows:
            break
        if i % 5000 == 0:
            print(f"    scanned {i:,}, kept {len(rows):,}", flush=True)

    return rows


def build_dataset(root: Path | None = None) -> tuple[list[dict], list[dict]]:
    root = root or Path(__file__).parent

    max_rows = cfg.MAX_ROWS if cfg.MAX_ROWS is not None else DEFAULT_MAX_ROWS
    print(f"Loading SimpleStories (cap {max_rows:,} rows)...")
    all_rows = load_simple_stories(max_rows)
    print(f"  {len(all_rows):,} rows after filtering")

    before = len(all_rows)
    all_rows = dedupe_rows(all_rows)
    if before != len(all_rows):
        print(f"  deduplicated {before:,} -> {len(all_rows):,} rows")

    if not all_rows:
        raise SystemExit(f"No rows loaded from {DATASET_ID}")

    random.seed(cfg.RANDOM_SEED)
    random.shuffle(all_rows)

    val_size = max(1, int(len(all_rows) * cfg.VAL_FRACTION))
    if cfg.MAX_VAL_SAMPLES:
        val_size = min(val_size, cfg.MAX_VAL_SAMPLES)
    val_rows = all_rows[:val_size]
    train_rows = all_rows[val_size:]

    if cfg.MAX_TRAIN_SAMPLES:
        train_rows = train_rows[: cfg.MAX_TRAIN_SAMPLES]

    return train_rows, val_rows


def prepare_data(root: Path | None = None) -> tuple[Path, Path]:
    root = root or Path(__file__).parent
    train_rows, val_rows = build_dataset(root)
    train_path = root / "data" / TRAIN_JSONL
    val_path = root / "data" / VAL_JSONL
    export_jsonl(train_rows, train_path)
    export_jsonl(val_rows, val_path)
    print(f"Wrote {len(train_rows):,} train -> {train_path}")
    print(f"Wrote {len(val_rows):,} val   -> {val_path}")
    return train_path, val_path


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Prepare SimpleStories (SimpleStories/SimpleStories on Hugging Face)"
    )
    parser.add_argument(
        "--max-rows",
        type=int,
        default=DEFAULT_MAX_ROWS,
        help=f"Stop after this many usable rows (default: {DEFAULT_MAX_ROWS:,})",
    )
    parser.add_argument(
        "--quick",
        action="store_true",
        help="Use only 500 rows for a fast smoke test",
    )
    args = parser.parse_args()
    if args.quick:
        cfg.MAX_ROWS = 500
        cfg.MAX_TRAIN_SAMPLES = 450
        cfg.MAX_VAL_SAMPLES = 50
    else:
        cfg.MAX_ROWS = args.max_rows
        cfg.MAX_TRAIN_SAMPLES = None
        cfg.MAX_VAL_SAMPLES = None
    prepare_data()


### `train.py`

In [ ]:
"""
Fine-tune BERT Encoder–Decoder for chapter title generation.

  python data_cmu_book_summaries.py    # download CMU Book Summaries
  python train.py   # fine-tune bert2bert
"""

from __future__ import annotations

import datasets  # noqa: F401 — must import before torch on Windows (pyarrow/CUDA)

import json
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from data_cmu_book_summaries import TRAIN_JSONL, VAL_JSONL, prepare_data
from config import (
    BATCH_SIZE,
    CHECKPOINT_DIR,
    DECODER_MODEL,
    ENCODER_MODEL,
    FP16,
    GRAD_ACCUM_STEPS,
    GRADIENT_CHECKPOINTING,
    LEARNING_RATE,
    MAX_INPUT_LENGTH,
    MAX_TARGET_LENGTH,
    NUM_EPOCHS,
    WEIGHT_DECAY,
    WARMUP_RATIO,
)


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def preprocess_rows(rows: list[dict], tokenizer, max_input: int, max_target: int):
    inputs = [r["text"] for r in rows]
    targets = [r["title"] for r in rows]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=max_target,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


class TitleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}


def load_tokenizer():
    return AutoTokenizer.from_pretrained(ENCODER_MODEL)


def build_model(tokenizer) -> EncoderDecoderModel:
    model = EncoderDecoderModel.from_encoder_decoder_pretrained(
        ENCODER_MODEL,
        DECODER_MODEL,
    )
    model.config.decoder_start_token_id = tokenizer.cls_token_id
    model.config.eos_token_id = tokenizer.sep_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
    model.generation_config.eos_token_id = tokenizer.sep_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.max_length = MAX_TARGET_LENGTH
    model.generation_config.num_beams = 4

    if GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    return model


def main():
    root = Path(__file__).parent
    train_path = root / "data" / TRAIN_JSONL
    val_path = root / "data" / VAL_JSONL

    if not train_path.exists():
        print("Preparing datasets...")
        prepare_data(root)

    train_rows = load_jsonl(train_path)
    val_rows = load_jsonl(val_path)
    if not train_rows:
        raise SystemExit("No training data. Run: python data_cmu_book_summaries.py")

    if torch.cuda.is_available():
        print(f"Device: cuda ({torch.cuda.get_device_name(0)})")
    else:
        print("Device: cpu (CUDA not found — install GPU PyTorch)")
        print("  pip install torch --index-url https://download.pytorch.org/whl/cu124")

    print(f"Train: {len(train_rows):,}  Val: {len(val_rows):,}")

    tokenizer = load_tokenizer()
    model = build_model(tokenizer)

    train_enc = preprocess_rows(train_rows, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH)
    val_enc = preprocess_rows(val_rows, tokenizer, MAX_INPUT_LENGTH, MAX_TARGET_LENGTH)

    train_ds = TitleDataset(train_enc)
    val_ds = TitleDataset(val_enc)
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    use_fp16 = FP16 and torch.cuda.is_available()
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(root / CHECKPOINT_DIR),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        fp16=use_fp16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        logging_steps=50,
        save_total_limit=2,
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        processing_class=tokenizer,
    )

    print("Starting fine-tune...")
    trainer.train()
    best_dir = root / CHECKPOINT_DIR / "best"
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    print(f"Saved to {best_dir}")


if __name__ == "__main__":
    main()


### `generate_title.py`

In [ ]:
"""
Generate chapter title candidates from narrative text.

  python generate_title.py --text "Once upon a time..."
  python generate_title.py --chapter-file chapter.txt
  python generate_title.py --eval
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, EncoderDecoderModel, LogitsProcessor, LogitsProcessorList

from config import (
    CHECKPOINT_DIR,
    CHUNK_CHARS,
    INPUT_PREFIX,
    MAX_GEN_LENGTH,
    MAX_INPUT_LENGTH,
    NUM_BEAMS,
    NUM_CANDIDATES,
)
from data_cmu_book_summaries import VAL_JSONL
from data_utils import chunk_text
from safety import blocked_word_token_ids


class BlockedWordsLogitsProcessor(LogitsProcessor):
    def __init__(self, blocked_ids: set[int]):
        self.blocked_ids = blocked_ids

    def __call__(self, input_ids, scores):
        if not self.blocked_ids:
            return scores
        for idx in self.blocked_ids:
            if idx < scores.shape[-1]:
                scores[:, idx] = float("-inf")
        return scores


def load_model_and_tokenizer(root: Path, checkpoint: Path | None):
    ckpt = checkpoint or (root / CHECKPOINT_DIR / "best")
    if not ckpt.exists():
        raise SystemExit(
            f"No checkpoint at {ckpt}. Run: python data_cmu_book_summaries.py && python train.py"
        )
    tokenizer = AutoTokenizer.from_pretrained(ckpt)
    model = EncoderDecoderModel.from_pretrained(ckpt)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return model, tokenizer, device


def format_input(text: str) -> str:
    return INPUT_PREFIX + chunk_text(text.strip(), CHUNK_CHARS)


def generate_titles(
    model,
    tokenizer,
    text: str,
    device: str,
    *,
    num_candidates: int = NUM_CANDIDATES,
    num_beams: int = NUM_BEAMS,
) -> list[str]:
    inputs = tokenizer(
        format_input(text),
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    blocked = blocked_word_token_ids(tokenizer)
    processors = LogitsProcessorList([BlockedWordsLogitsProcessor(blocked)])

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_GEN_LENGTH,
            num_beams=max(num_beams, num_candidates),
            num_return_sequences=num_candidates,
            early_stopping=True,
            logits_processor=processors,
        )

    candidates: list[str] = []
    seen: set[str] = set()
    for seq in outputs:
        title = tokenizer.decode(seq, skip_special_tokens=True).strip()
        key = title.lower()
        if title and key not in seen:
            seen.add(key)
            candidates.append(title)
    return candidates


def main():
    parser = argparse.ArgumentParser(description="Generate chapter title candidates")
    parser.add_argument("--text", type=str, help="Chapter / story text")
    parser.add_argument("--chapter-file", type=Path, help="File with chapter text")
    parser.add_argument("--eval", action="store_true", help="Evaluate on val set")
    parser.add_argument("--checkpoint", type=Path, default=None)
    parser.add_argument("--candidates", type=int, default=NUM_CANDIDATES)
    args = parser.parse_args()

    root = Path(__file__).parent
    model, tokenizer, device = load_model_and_tokenizer(root, args.checkpoint)

    if args.eval:
        val_path = root / "data" / VAL_JSONL
        if not val_path.exists():
            raise SystemExit("No validation file. Run: python data_cmu_book_summaries.py")

        exact = total = 0
        with val_path.open(encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                text = row.get("text", "")
                if text.startswith(INPUT_PREFIX):
                    text = text[len(INPUT_PREFIX) :]
                gold = row["title"]
                preds = generate_titles(
                    model, tokenizer, text, device, num_candidates=1
                )
                pred = preds[0] if preds else ""
                match = pred.strip().lower() == gold.strip().lower()
                exact += int(match)
                total += 1
                mark = "OK" if match else "  "
                print(f"{mark} {gold}")
                if not match:
                    print(f"     -> {pred}")
        print(f"\nExact match: {exact}/{total}")
        return

    if args.chapter_file:
        text = args.chapter_file.read_text(encoding="utf-8")
    elif args.text:
        text = args.text
    else:
        parser.error("Provide --text, --chapter-file, or --eval")

    titles = generate_titles(
        model, tokenizer, text, device, num_candidates=args.candidates
    )
    print("Suggested titles:")
    for i, t in enumerate(titles, 1):
        print(f"  {i}. {t}")


if __name__ == "__main__":
    main()


## Usage (from repository root)

```powershell
pip install -r requirements.txt

# Prepare datasets
python data_cmu_book_summaries.py
python data_novel_chapter.py
python data_simple_stories.py

# Train (CMU default)
python train.py

# Generate titles
python generate_title.py --text "Your chapter passage here..."
python generate_title.py --eval
```


In [ ]:
# Optional demo cell (requires trained checkpoint in checkpoints/bert2bert-titles/best/)
from pathlib import Path

sample = (
    "The old lighthouse keeper climbed the spiral stairs for the last time. "
    "Storm clouds gathered over the harbor as ships raced for shore. "
    "He had kept the light burning for forty years, but tonight the bulb "
    "would go dark forever."
)

checkpoint = Path("checkpoints/bert2bert-titles/best")
if checkpoint.exists():
    from generate_title import generate_titles, load_model_and_tokenizer

    root = Path(".")
    model, tokenizer, device = load_model_and_tokenizer(root)
    titles = generate_titles(model, tokenizer, sample, device)
    print("Sample passage:")
    print(sample)
    print("\nSuggested titles:")
    for i, t in enumerate(titles, 1):
        print(f"  {i}. {t}")
else:
    print("No checkpoint found. Run data prep and train.py first.")